# Clustering Comparison: Wine Dataset (model zoo)

Dataset: UCI Wine, loaded with `sc4020.datasets.load_wine()` (178 samples, 13 continuous
chemical properties, standardized). Ground truth is the 3 wine cultivars.

Seven clustering algorithms are compared with internal metrics (silhouette, Davies-Bouldin)
and an external metric (ARI vs. the true cultivars).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_wine
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "ari",
]

## Load and standardize

In [ ]:
ds = load_wine()
print(f"{ds.name}: X={ds.data.shape}, classes={ds.target_names}, n_true_clusters={ds.n_true_clusters}")

## 2D embedding for visualization

Clustering runs on the full standardized 13-D space; PCA to 2D is *only* used to plot.

In [ ]:
X_2d, xlabel, ylabel = gf.project_2d(ds.data)
print(xlabel, ylabel)

## Fit 7 clustering models

* **By the shared rule** (`tuning.select_all`): K-Means and Gaussian Mixture at k=3 (GMM covariance
  type by BIC); DBSCAN and HDBSCAN at the setting with the best silhouette among those with at least
  2 clusters and at most 50% noise.
* **Hierarchical:** Agglomerative (k=3, Ward linkage)
* **Graph-based:** Spectral (k=3, nearest-neighbours affinity)
* **Exemplar-based:** Affinity Propagation (damping=0.8)

In [ ]:
# K-Means, DBSCAN, HDBSCAN and GMM by the shared rule (tuning.select_all), with K-Means and GMM at
# the number of classes; Agglomerative (Ward) and Spectral at the same k; Affinity Propagation
# picks its own number of exemplars.
k = ds.n_true_clusters
selections = tuning.select_all(ds.data, k=k, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Agglomerative"] = {"n_clusters": k, "linkage": "ward"}
params["Spectral"] = {"n_clusters": k, "affinity": "nearest_neighbors"}
params["Affinity Propagation"] = {"damping": 0.8}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

In [ ]:
results_df = gf.build_results_table("wine_model_zoo", ds.data, results, ds.target)
display(results_df[METRIC_COLS].round(4))
results_df.to_csv(results_dir("datasets") / "wine_model_zoo_results.csv", index=False)

## Cluster visualizations vs. ground truth

In [ ]:
gf.plot_cluster_grid(
    ds.data, results, y_true=ds.target, class_names=ds.target_names, scores=gf.score_labels(results_df),
    title="Ground truth and 7 clustering methods (Wine)", ncols=4,
)
plt.show()

## Discussion

* **Top performers (K-Means & GMM):** standardized chemical properties form three convex,
  globular clusters, so both recover the cultivars almost perfectly (ARI ≈ 0.8975).
* **DBSCAN limitation:** there is no strict density gap between cultivars. A single global
  radius ε merges adjacent cultivars while treating peripheral samples as noise.
* **Affinity Propagation over-segmentation:** with no cluster-count constraint, message
  passing splits the 3 cultivars into many micro-clusters.